In [0]:
from pyspark.sql.functions import to_date,col,desc,dense_rank, count
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank,avg, asc

In [0]:
df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/default/training/Sales.csv")
df.display()
df.printSchema()

In [0]:
df1 = df.withColumnRenamed("Product_id", "dept")
df1.display()

In [0]:
from pyspark.sql.functions import when, col

df_updated = df1.withColumn('dept', when(col('dept').isin([103, 102,105,104]), 101).otherwise(col('dept')) )\
    .withColumn('price', when(col('price').isin([300,500]),100).otherwise(col('price')))\
    .withColumn('price', when(col('price').isin([120]),50).otherwise(col('price')))
df_updated.display()

TOP N salary use dense rank

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, desc

window = Window.orderBy(desc('price'))
df2 = df1.withColumn('rank', dense_rank().over(window)).filter(col('rank') == 2)
df2.display()

In [0]:
df3 = df_updated.withColumn('rank', rank().over(
    Window.partitionBy('dept').orderBy(desc('price'))))\
        .withColumn('denserank', dense_rank().over(Window.partitionBy('dept').orderBy(desc('price'))))\
        .withColumn('row_number', row_number().over(Window.partitionBy('dept').orderBy(desc('price'))))
        
    
df3.display()

find and delete duplicate records use rownum or groupby having 

In [0]:
from pyspark.sql.functions import asc

dfdup = df_updated.withColumn('duplicate', row_number().over(
    Window.partitionBy('price').orderBy(asc('price'))
)).filter(col('duplicate') == 1)
dfdup.display()

In [0]:
from pyspark.sql.functions import count, col

df2_grouped = df_updated.groupBy(col("price")).agg(count(col("price")).alias("pricecount")) \
                .filter(col("pricecount") >= 1)
df2_grouped.display()

fetch salary greater than avg

In [0]:
df.createOrReplaceTempView("df_table")

In [0]:
df_sql_avg = spark.sql("""
    SELECT *
    FROM df_table
    WHERE price > (SELECT AVG(price) as A FROM df_table)
""")

df_sql_avg.display()

In [0]:
avg_salary = df.select(avg("price")).collect()[0][0]
print(avg_salary)

In [0]:
df_above_avg = df.filter(df.price > avg_salary)
df_above_avg.show()

top 5


In [0]:
df.orderBy(col("price").desc()).limit(4).display()

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("EmployeeDF").getOrCreate()

# Employee data
data = [
    (1, "Rahul", "IT", 70000),
    (2, "Neha", "HR", 60000),
    (3, "Amit", "IT", 90000),
    (4, "Priya", "Finance", 80000),
    (5, "Karan", "IT", 90000),
    (6, "Sneha", "HR", 65000)
]

# Column names
columns = ["emp_id", "emp_name", "department", "salary"]

# Create DataFrame
dfemp = spark.createDataFrame(data, columns)

dfemp.display()


How to find the department with the highest number of employees?

In [0]:
#use limit
from pyspark.sql.functions import count, col

dfemp.groupBy("department").agg(count("emp_id")).orderBy(col("count(emp_id)").desc()).display()

In [0]:
from pyspark.sql.functions import count,max

dfe=dfemp.groupBy("department").agg(count("emp_id").alias("count"))
dfe.display()
dfe.agg(max("count")).display()

In [0]:
dfemp.display()

Write a query to fetch employees having the highest salary in each
department.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col

dfemp.withColumn('rank',dense_rank().over(Window.partitionBy("department").orderBy(col("salary").desc()))).filter(col('rank')==1).display()

calculation

In [0]:
dfmult = dfemp.withColumn('salary', col('salary') * 2).display()
dfs = dfemp.filter((col('salary') >= 60000) & (col('salary') <= 80000)).display()

 How to find the department with the lowest average salary?

In [0]:
dfemp.groupBy('department').agg(avg('salary').alias('sal')).orderBy(col('sal')).limit(1).display()

How to find the emp with the lowest average salary in each department

In [0]:
from pyspark.sql.functions import avg, dense_rank, col
from pyspark.sql.window import Window

In [0]:
from pyspark.sql.functions import avg, col
from pyspark.sql.window import Window

dl1 =dfemp.withColumn(
    "dept_avg_salary",
    avg("salary").over(Window.partitionBy("department"))
).filter(col("salary") >= col("dept_avg_salary")).display()


drop pparticular row

In [0]:
df_drop=dfemp.filter(col('salary') != 7000).display()

In [0]:
dfemp.write.mode("overwrite").saveAsTable("employee")

In [0]:
df_with_avg = df.withColumn("avgprice", avg("price").over(window.orderBy("price")))
df_with_avg.display()

# Filter employees whose salary > department average


In [0]:
result_df = df_with_avg.filter(col("price") > col("avgprice"))